# Qwen2.5-3B Mega-Calibration (Colab)

**One Colab run → calibration data for 4 downstream dismantle wins:**
1. **Eagle5 v2 head training** (layer-32 residual + intermediate)
2. **AWQ smoothing factors** (per-channel mean|x| × all layers × 7 sites)
3. **Per-channel W4A8 static scales** (per-channel max|x| × all layers × 7 sites)
4. **Quality benchmarks ground truth** (top-100 logits per token)

**Compute:** ~6-8 hr on H100, ~$8-12 in Pro compute units. Fits monthly Pro budget.

**Output:** parquet shards on Drive + 1 resumable activation-stats .npz file.

In [ ]:
# Cell 1 — Mount Drive + verify GPU
from google.colab import drive
drive.mount('/content/drive')

import torch
assert torch.cuda.is_available(), 'No CUDA — set Runtime → Change runtime type → GPU'
name = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {name}  VRAM: {vram:.1f} GB')

In [ ]:
# Cell 2 — Install deps
!pip install -q 'accelerate>=1.0' 'datasets>=3.0' 'pyarrow>=17' 'tqdm>=4.66' 'zstandard' 'bitsandbytes>=0.43'
import transformers, datasets, pyarrow, accelerate
print(f'transformers={transformers.__version__}  datasets={datasets.__version__}')
print(f'pyarrow={pyarrow.__version__}  accelerate={accelerate.__version__}')

In [ ]:
# Cell 3 — Clone/update dismantle
import os
REPO = 'https://github.com/joshuahickscorp/dismantle.git'
BRANCH = 'main'
if not os.path.exists('/content/dismantle'):
    !cd /content && git clone --depth 1 --branch {BRANCH} {REPO}
else:
    !cd /content/dismantle && git fetch origin {BRANCH} --depth 1 && git checkout {BRANCH} && git reset --hard origin/{BRANCH}
%cd /content/dismantle
!git rev-parse --short HEAD
!ls -l colab/mega_calibrate.py

In [ ]:
# Cell 4 — RUN MEGA-CALIBRATION
#
# Qwen2.5-3B-Instruct (dismantle's primary product target).
# 2000 prompts × 2048 tokens, capture-layer=32 (of 36, near-top).
#
# Auto-detects GPU:
#   >=70 GB (G4/H100/Blackwell): native fp16, batch 8
#   >=35 GB (A100 40):           native fp16, batch 6
#   >=20 GB (L4 24GB):           4-bit nf4, batch 4
#   <20 GB (T4/V100):            4-bit nf4, batch 2
#
# Resumable: rerun the cell after any disconnect — Drive shards and
# activation stats are used to resume safely.

import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

OUT = '/content/qwen3b_corpus'  # local SSD during each shard
SYNC_DIR = '/content/drive/MyDrive/dismantle/qwen3b_corpus'
os.makedirs(OUT, exist_ok=True)
os.makedirs(SYNC_DIR, exist_ok=True)

import torch
gpu_name = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
if vram >= 70:
    tier = 'G4/H100/Blackwell 70GB+'
    load_flag = ''
    batch = 8
    lm_head_chunk = 256
elif vram >= 35:
    tier = 'A100 40GB-class'
    load_flag = ''
    batch = 6
    lm_head_chunk = 128
elif vram >= 20:
    tier = 'L4 24GB-class'
    load_flag = '--load-4bit'
    batch = 4
    lm_head_chunk = 64
else:
    tier = 'T4/V100 16GB-class'
    load_flag = '--load-4bit'
    batch = 2
    lm_head_chunk = 32
print(f'GPU={gpu_name}  VRAM={vram:.0f} GB  tier={tier}')
print(f'→ batch={batch}  lm_head_chunk={lm_head_chunk}  load={"4-bit" if load_flag else "fp16"}')

!python colab/mega_calibrate.py \
  --model Qwen/Qwen2.5-3B-Instruct \
  --max-sequences 2000 \
  --max-tokens 2048 \
  --batch-size {batch} \
  --capture-layer 32 \
  --topk-logits 100 \
  --lm-head-chunk-tokens {lm_head_chunk} \
  --shard-size 16 \
  --sync-dir {SYNC_DIR} \
  --sync-every 4 \
  --delete-local-after-sync \
  {load_flag} \
  --out {OUT}

In [ ]:
# Cell 5 — Final verify/sync corpus + activation stats → Drive
import os, glob, shutil, time
SRC = '/content/qwen3b_corpus'
DST = '/content/drive/MyDrive/dismantle/qwen3b_corpus'
os.makedirs(DST, exist_ok=True)

for f in sorted(glob.glob(f'{SRC}/*')):
    name = os.path.basename(f)
    d = f'{DST}/{name}'
    if not os.path.exists(d) or os.path.getsize(d) != os.path.getsize(f):
        shutil.copy2(f, d)

shards = sorted(glob.glob(f'{DST}/*.parquet'))
total_mb = sum(os.path.getsize(f) for f in shards) / 1e6
stats_path = f'{DST}/per_site_activation_stats.npz'
stats_mb = os.path.getsize(stats_path) / 1e6 if os.path.exists(stats_path) else 0

print(f'\n✅ {len(shards)} shards on Drive, {total_mb:.0f} MB')
print(f'✅ activation stats: {stats_mb:.1f} MB at {stats_path}')
print()
print('Download to laptop:')
print('  1. drive.google.com → MyDrive/dismantle/qwen3b_corpus → Download')
print('  2. Unzip into artifacts/calibration/qwen3b_corpus/ on laptop')
print('  3. Run laptop post-processing:')
print('     - Train Qwen3B Eagle5 head (~2 hr MLX)')
print('     - Apply AWQ algorithm to per_site_activation_stats.npz (~30 min)')
print('     - Bench stacked configs')